# GRPO in miniature: "the group is the baseline"

**A tiny contextual-bandit demo of GRPO's group-relative advantage — no LLM, no gym, CPU-only.**

This is a sibling to the simple-RL companion notebook. Here we isolate the *one*
idea that makes **GRPO** (Group Relative Policy Optimization) different from
vanilla policy gradient and from actor-critic: **how you compute the credit /
baseline**.

## The shared policy gradient

All three methods we compare optimize the *same* objective with the *same*
score-function (REINFORCE) estimator:

$$\nabla_\theta J(\theta) \;=\; \mathbb{E}_{a\sim\pi_\theta}\big[\,\nabla_\theta \log \pi_\theta(a\mid c)\;\cdot\;\Phi\,\big].$$

Only the **weight** $\Phi$ (the "credit" assigned to a sampled action) differs:

| Method | weight $\Phi$ | needs a critic? |
|---|---|---|
| **REINFORCE (no baseline)** | $r$ (raw reward) | no |
| **Value-baseline PG (A2C/PPO-style)** | $r - V(c)$ (advantage) | **yes**, learn $V(c)$ |
| **GRPO** | $\hat A_i = \dfrac{r_i - \mathrm{mean}(r_{\text{group}})}{\mathrm{std}(r_{\text{group}}) + \varepsilon}$ | **no** — the group mean *is* the baseline |

### Why a baseline is legal (the EGLP lemma)

The **Expected Grad-Log-Prob** lemma says $\mathbb{E}_{a\sim\pi}[\nabla_\theta\log\pi_\theta(a\mid c)\,b(c)] = b(c)\,\nabla_\theta\sum_a\pi_\theta(a\mid c) = b(c)\,\nabla_\theta 1 = 0$
for **any** baseline $b(c)$ that does **not depend on the sampled action** $a$.
So subtracting a per-context baseline (a learned $V(c)$, *or* the mean reward of
a freshly-sampled group) leaves the gradient **unbiased** while slashing its
**variance**. GRPO's group mean is exactly such an action-independent baseline,
estimated on the fly from $G$ samples — **no value network required.**

### The bandit as a stand-in for prompt → response → reward

A **contextual bandit** is the minimal RL setting that already contains the LLM-RL
loop:

* **context** $c$  ≈  *prompt*
* **action** $a$ (one of $K$)  ≈  *response / completion*
* **reward** $r(c,a)$  ≈  *reward-model / verifier score (RLVR)*

There is no multi-step credit assignment, no environment dynamics, no value
bootstrap — just `prompt → sample response → score`. That is **exactly** the
structure GRPO was designed for, which is why we can show the whole idea in a
single CPU notebook with no LLM.


In [ ]:
# If running on a fresh Colab and matplotlib is somehow missing, uncomment:
# %pip install -q matplotlib
import time, math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

print("numpy", np.__version__, "| torch", torch.__version__)
print("device: cpu (this whole notebook is tiny & CPU-only)")
_t_start = time.time()  # we print total elapsed time at the very end


In [ ]:
# ============================ CONFIG =====================================
# Everything is deliberately tiny so the notebook finishes in << 1 minute.
N_CONTEXTS = 8       # number of "prompts"
K          = 10      # number of discrete actions ("answers") per context
GROUP_SIZE = 8       # G: how many actions GRPO samples per context per step
ITERS      = 400     # optimization steps
LR         = 0.5     # learning rate (tabular logits tolerate a big LR)
SEEDS      = [0, 1, 2, 3, 4]   # we average learning curves over these seeds
REWARD_NOISE = 0.10  # std of Gaussian noise added to each observed reward
EPS        = 1e-8    # numerical floor for the GRPO std normalization
# For the value-baseline critic we use an EMA estimate of V(c):
V_EMA_BETA = 0.9
# =========================================================================
print(f"{N_CONTEXTS} contexts x {K} actions | group size G={GROUP_SIZE} | "
      f"{ITERS} iters | {len(SEEDS)} seeds")


## 1. The reward table $r(c,a)$ and the optimal-return reference

We fix a reward table of shape `[N_CONTEXTS, K]`. Think of each row as "for this
prompt, how good is each of the $K$ candidate answers." We add small Gaussian
noise at sampling time (like a noisy reward model / verifier). The **optimal
policy** deterministically picks the best action per context; its expected return
is the mean of the per-context maxima — our dashed reference line in the plots.


In [ ]:
def make_reward_table(seed=12345):
    '''Fixed, mildly-structured reward table in [0, 1], shape [N_CONTEXTS, K].

    We build a smooth bump per context (so there is a clear best action and a
    gradient of near-good actions) plus a little per-(c,a) random structure.
    The *table* is fixed across all methods/seeds; only the OBSERVATION noise
    (REWARD_NOISE) varies per sample.
    '''
    rng = np.random.default_rng(seed)
    actions = np.arange(K)
    table = np.zeros((N_CONTEXTS, K), dtype=np.float64)
    for c in range(N_CONTEXTS):
        center = rng.integers(0, K)               # best action for this context
        width  = rng.uniform(1.5, 3.0)
        bump   = np.exp(-0.5 * ((actions - center) / width) ** 2)  # smooth peak
        noise  = 0.15 * rng.standard_normal(K)    # fixed per-(c,a) roughness
        row    = bump + noise
        row    = (row - row.min()) / (row.max() - row.min() + 1e-12)  # -> [0,1]
        table[c] = row
    return table

REWARD_TABLE = make_reward_table()
OPTIMAL_ACTIONS = REWARD_TABLE.argmax(axis=1)          # best answer per prompt
OPTIMAL_RETURN  = REWARD_TABLE.max(axis=1).mean()      # expected return of opt policy
print("reward table shape:", REWARD_TABLE.shape)
print("optimal action per context:", OPTIMAL_ACTIONS.tolist())
print(f"optimal expected return (reference line): {OPTIMAL_RETURN:.4f}")


def reward_fn(contexts, actions, rng):
    '''Observed (noisy) reward for arrays of contexts & actions. Vectorized.

    contexts, actions: integer arrays of the same shape.
    Returns float array of the same shape: table[c,a] + N(0, REWARD_NOISE^2).
    '''
    base = REWARD_TABLE[contexts, actions]
    return base + REWARD_NOISE * rng.standard_normal(size=base.shape)


## 2. The policy: per-context softmax over $K$ actions

The policy is a **tabular** logit matrix $\theta \in \mathbb{R}^{N\times K}$:
$\pi_\theta(a\mid c) = \mathrm{softmax}_a(\theta_{c,\cdot})$. (Equivalently, a
linear layer on a one-hot context — but tabular keeps the math transparent.)
We provide three helpers used by all three trainers:

* `policy_logprobs(theta)` → log-probabilities, shape `[N, K]`
* `sample_actions(theta, n, rng_torch)` → sample `n` actions per context
* `greedy_return(theta)` → the *deterministic* (argmax) return, our eval metric


In [ ]:
def new_policy(seed):
    '''Fresh tabular logits theta (requires grad). Small random init so every
    method starts from the SAME stochastic policy for a given seed.'''
    g = torch.Generator().manual_seed(seed)
    theta = 0.01 * torch.randn(N_CONTEXTS, K, generator=g)
    theta.requires_grad_(True)
    return theta

def policy_logprobs(theta):
    '''log pi(.|c) for every context, shape [N_CONTEXTS, K].'''
    return F.log_softmax(theta, dim=1)

def sample_actions(theta, n_per_context, gen):
    '''Sample n actions per context from the current policy.
    Returns LongTensor [N_CONTEXTS, n_per_context].'''
    probs = F.softmax(theta.detach(), dim=1)          # [N, K]
    return torch.multinomial(probs, n_per_context, replacement=True, generator=gen)

@torch.no_grad()
def greedy_return(theta):
    '''Deterministic eval: take argmax action per context, average true (noise-free)
    table reward. This is our 'mean reward vs iteration' learning-curve metric.'''
    greedy = theta.argmax(dim=1).cpu().numpy()
    return REWARD_TABLE[np.arange(N_CONTEXTS), greedy].mean()

@torch.no_grad()
def stochastic_return(theta):
    '''Expected return UNDER the current stochastic policy (no noise):
    sum_a pi(a|c) r(c,a), averaged over contexts. Smoother curve.'''
    probs = F.softmax(theta, dim=1).cpu().numpy()
    return float((probs * REWARD_TABLE).sum(axis=1).mean())


## 3. A low-variance "true" gradient (for the variance diagnostic)

To measure *gradient quality* we compare each method's noisy single-step gradient
estimate against a **low-variance reference gradient** computed in closed form.

For a softmax policy the exact policy gradient w.r.t. the logits is analytic:

$$\frac{\partial J}{\partial \theta_{c,a}} = \pi(a\mid c)\,\big(\bar r(c,a) - \mathbb{E}_{a'}[\bar r(c,a')]\big),\qquad \bar r(c,a)=\text{(noise-free) table reward}.$$

(The advantage form $\bar r - \mathbb{E}[\bar r]$ appears *automatically* — that
is the EGLP baseline showing up in the exact gradient.) We will measure, for each
method, the **cosine similarity** between its single-sample gradient estimate and
this true gradient. Higher cosine = lower-variance / better-aligned estimator.


In [ ]:
def true_gradient(theta):
    '''Exact dJ/dtheta for the softmax policy using the noise-free reward table.
    Returns a tensor [N_CONTEXTS, K]. Used ONLY as a reference for the cosine
    diagnostic (never to train).'''
    with torch.no_grad():
        probs = F.softmax(theta, dim=1)                       # [N, K]
        r = torch.tensor(REWARD_TABLE, dtype=probs.dtype)     # [N, K]
        baseline = (probs * r).sum(dim=1, keepdim=True)       # E_a[r] per context
        grad = probs * (r - baseline)                         # [N, K]
    return grad

def cosine(g1, g2):
    '''Flattened cosine similarity between two gradient tensors.'''
    a = g1.reshape(-1); b = g2.reshape(-1)
    return float(F.cosine_similarity(a, b, dim=0, eps=1e-12))


## 4. Three trainers — same gradient, three different $\Phi$

Each trainer runs one SGD loop and returns:

* `curve` — greedy return vs iteration (the learning curve), and
* `cos_list` — per-step cosine of its *single-sample* gradient estimate to the
  true gradient (our variance / quality diagnostic).

Read the three carefully: the **only** conceptual difference is the boxed weight
$\Phi$ fed into `(-logprob * Phi).sum()`. Everything else — sampling, the
score function $\nabla\log\pi$, the optimizer — is identical.


In [ ]:
def reinforce_nobaseline(seed):
    '''(1) REINFORCE with NO baseline:  Phi = raw reward r.  High variance.'''
    rng   = np.random.default_rng(1000 + seed)
    gen   = torch.Generator().manual_seed(2000 + seed)
    theta = new_policy(seed)
    opt   = torch.optim.SGD([theta], lr=LR)
    curve, cos_list = [], []
    for it in range(ITERS):
        # sample ONE action per context from the current policy
        actions = sample_actions(theta, 1, gen)[:, 0]                 # [N]
        ctx     = torch.arange(N_CONTEXTS)
        r       = torch.tensor(reward_fn(ctx.numpy(), actions.numpy(), rng),
                               dtype=torch.float32)                   # [N]
        # ---- the weight Phi : raw reward, NO baseline ----
        phi     = r                                                   # <<<<<<
        logp    = policy_logprobs(theta)[ctx, actions]                # [N]
        loss    = -(logp * phi).mean()
        opt.zero_grad(); loss.backward()
        # record cosine of THIS estimate to the true gradient before stepping
        # optimizer ascends along -grad(loss); compare THAT to the true ascent
        # gradient so 'better aligned' => higher (positive) cosine.
        cos_list.append(cosine(-theta.grad.detach(), true_gradient(theta)))
        opt.step()
        curve.append(greedy_return(theta))
    return np.array(curve), np.array(cos_list)


In [ ]:
def value_baseline(seed):
    '''(2) Value-baseline PG (A2C/PPO-style critic):  Phi = r - V(c).
    Here V(c) is a per-context EMA estimate of the reward — a stand-in for a
    learned scalar critic. This is the classic actor-critic baseline.'''
    rng   = np.random.default_rng(1000 + seed)
    gen   = torch.Generator().manual_seed(2000 + seed)
    theta = new_policy(seed)
    opt   = torch.optim.SGD([theta], lr=LR)
    V     = np.full(N_CONTEXTS, 0.5, dtype=np.float64)   # the "critic" (EMA of r)
    curve, cos_list = [], []
    for it in range(ITERS):
        actions = sample_actions(theta, 1, gen)[:, 0]
        ctx     = torch.arange(N_CONTEXTS)
        r_np    = reward_fn(ctx.numpy(), actions.numpy(), rng)        # [N]
        # ---- the weight Phi : advantage = reward - learned value baseline ----
        adv     = r_np - V                                            # <<<<<<
        phi     = torch.tensor(adv, dtype=torch.float32)
        logp    = policy_logprobs(theta)[ctx, actions]
        loss    = -(logp * phi).mean()
        opt.zero_grad(); loss.backward()
        # optimizer ascends along -grad(loss); compare THAT to the true ascent
        # gradient so 'better aligned' => higher (positive) cosine.
        cos_list.append(cosine(-theta.grad.detach(), true_gradient(theta)))
        opt.step()
        # update the critic AFTER using it (EMA toward observed reward)
        V = V_EMA_BETA * V + (1 - V_EMA_BETA) * r_np
        curve.append(greedy_return(theta))
    return np.array(curve), np.array(cos_list)


In [ ]:
def grpo(seed):
    '''(3) GRPO:  for each context sample a GROUP of G actions, and use the
    GROUP-RELATIVE advantage as Phi:

            A_hat_i = (r_i - mean(r_group)) / (std(r_group) + eps)

    *** THE GROUP MEAN IS THE BASELINE. *** No value network is trained.
    Because mean(r_group) is computed from freshly-sampled actions, it does NOT
    depend on any individual sampled action -> EGLP says it is an UNBIASED
    baseline. Dividing by the group std just rescales/normalizes the advantage
    (per-prompt whitening) -> stable step sizes across prompts.
    '''
    rng   = np.random.default_rng(1000 + seed)
    gen   = torch.Generator().manual_seed(2000 + seed)
    theta = new_policy(seed)
    opt   = torch.optim.SGD([theta], lr=LR)
    curve, cos_list = [], []
    ctx_col = torch.arange(N_CONTEXTS).unsqueeze(1).expand(N_CONTEXTS, GROUP_SIZE)
    for it in range(ITERS):
        # sample a GROUP of G actions per context
        actions = sample_actions(theta, GROUP_SIZE, gen)             # [N, G]
        r_np    = reward_fn(ctx_col.numpy(), actions.numpy(), rng)   # [N, G]
        # ---- group-relative advantage: subtract per-row mean, divide by row std ----
        mean    = r_np.mean(axis=1, keepdims=True)                   # baseline = group mean
        std     = r_np.std(axis=1, keepdims=True)
        adv     = (r_np - mean) / (std + EPS)                        # <<<<<< A_hat, [N, G]
        phi     = torch.tensor(adv, dtype=torch.float32)
        # score function for every (context, sampled action) in the group
        logp    = policy_logprobs(theta).gather(1, actions)          # [N, G]
        loss    = -(logp * phi).mean()
        opt.zero_grad(); loss.backward()
        # optimizer ascends along -grad(loss); compare THAT to the true ascent
        # gradient so 'better aligned' => higher (positive) cosine.
        cos_list.append(cosine(-theta.grad.detach(), true_gradient(theta)))
        opt.step()
        curve.append(greedy_return(theta))
    return np.array(curve), np.array(cos_list)


### Sanity check: the group-relative advantage has ~zero mean per group

The whole point of "the group mean is the baseline" is that, within each group,
the centered advantage sums to ~zero (and after dividing by std, has ~unit
scale). Let's verify on one freshly-sampled batch.


In [ ]:
_gen = torch.Generator().manual_seed(0)
_rng = np.random.default_rng(0)
_theta = new_policy(0)
_ctx = torch.arange(N_CONTEXTS).unsqueeze(1).expand(N_CONTEXTS, GROUP_SIZE)
_a = sample_actions(_theta, GROUP_SIZE, _gen)
_r = reward_fn(_ctx.numpy(), _a.numpy(), _rng)
_adv = (_r - _r.mean(1, keepdims=True)) / (_r.std(1, keepdims=True) + EPS)
print("per-group mean of advantage (should be ~0):", np.round(_adv.mean(axis=1), 6))
print("max |per-group mean| =", np.abs(_adv.mean(axis=1)).max())
assert np.abs(_adv.mean(axis=1)).max() < 1e-6, "group advantage must be ~zero-mean!"
print("OK: group-relative advantage is zero-mean per prompt -> the group IS the baseline.")


## 5. Run all three methods across seeds

In [ ]:
METHODS = {
    "REINFORCE (no baseline)": reinforce_nobaseline,
    "Value baseline (A2C/PPO)": value_baseline,
    "GRPO (group baseline)": grpo,
}
results = {}   # name -> dict(curves=[seeds,iters], cos=[seeds,iters])
t0 = time.time()
for name, fn in METHODS.items():
    curves, coss = [], []
    for s in SEEDS:
        c, cos = fn(s)
        curves.append(c); coss.append(cos)
    results[name] = {"curves": np.stack(curves), "cos": np.stack(coss)}
    final = results[name]["curves"][:, -1]
    print(f"{name:28s} final greedy return = {final.mean():.4f} +/- {final.std():.4f}")
print(f"\ntrained all 3 methods x {len(SEEDS)} seeds in {time.time()-t0:.2f}s")


## 6. Plot 1 — learning curves (mean ± std over seeds)

All three climb toward the optimal dashed line. GRPO and the value-baseline are
typically faster / steadier than baseline-free REINFORCE — **but GRPO got there
without ever training a critic.**


In [ ]:
plt.figure(figsize=(7.5, 4.5))
colors = {"REINFORCE (no baseline)": "tab:red",
          "Value baseline (A2C/PPO)": "tab:orange",
          "GRPO (group baseline)": "tab:blue"}
xs = np.arange(ITERS)
for name, res in results.items():
    m = res["curves"].mean(0); sd = res["curves"].std(0)
    plt.plot(xs, m, label=name, color=colors[name], lw=2)
    plt.fill_between(xs, m - sd, m + sd, color=colors[name], alpha=0.15)
plt.axhline(OPTIMAL_RETURN, ls="--", color="k", lw=1, label=f"optimal = {OPTIMAL_RETURN:.3f}")
plt.xlabel("iteration"); plt.ylabel("greedy return (mean true reward)")
plt.title("Learning curves: same policy gradient, three baselines")
plt.legend(loc="lower right", fontsize=9); plt.grid(alpha=0.3); plt.tight_layout()
plt.show()


## 7. Plot 2 — gradient quality (cosine to the true gradient)

This is the *real* reason baselines help. We plot the per-step cosine similarity
between each method's single-batch gradient estimate and the exact gradient
(higher = lower variance, better aligned). The bar chart shows the average over
all steps & seeds.

* **REINFORCE (no baseline)** — noisy, low cosine.
* **Value baseline** — subtracting $V(c)$ removes the action-independent part →
  higher cosine.
* **GRPO** — the group mean baseline + per-prompt std-normalization →
  comparable or better cosine, with **no critic**, and it *improves as $G$ grows*.


In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.3))
# Left: smoothed cosine vs iteration (moving average for readability)
def smooth(x, w=15):
    if len(x) < w: return x
    k = np.ones(w) / w
    return np.convolve(x, k, mode="valid")
for name, res in results.items():
    cos_mean = res["cos"].mean(0)            # avg over seeds, per iteration
    axL.plot(smooth(cos_mean), label=name, color=colors[name], lw=2)
axL.set_xlabel("iteration (smoothed)"); axL.set_ylabel("cosine(estimate, true grad)")
axL.set_title("Gradient alignment vs iteration"); axL.grid(alpha=0.3)
axL.legend(fontsize=8, loc="lower right")
# Right: mean cosine bar chart (over all steps & seeds)
names = list(results.keys())
means = [results[n]["cos"].mean() for n in names]
stds  = [results[n]["cos"].std()  for n in names]
bars  = axR.bar(range(len(names)), means, yerr=stds, capsize=5,
                color=[colors[n] for n in names], alpha=0.85)
axR.set_xticks(range(len(names)))
axR.set_xticklabels(["REINFORCE\n(no baseline)", "Value\nbaseline", "GRPO\n(group)"])
axR.set_ylabel("mean cosine to true gradient")
axR.set_title("Average gradient quality (higher = lower variance)")
axR.grid(alpha=0.3, axis="y")
for b, m in zip(bars, means):
    axR.text(b.get_x() + b.get_width()/2, m, f"{m:.3f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

print("mean cosine-to-true-gradient (higher is better, lower variance):")
for n in names:
    print(f"  {n:28s} {results[n]['cos'].mean():.4f}")


## 8. Bonus — GRPO's variance shrinks as the group size $G$ grows

The group-mean baseline is estimated from $G$ samples; bigger $G$ → a tighter
baseline → lower-variance advantage → better gradient alignment. Quick sweep:


In [ ]:
def grpo_cos_for_G(G, seed=0, steps=120):
    '''Average gradient cosine for GRPO with a given group size G.'''
    rng = np.random.default_rng(1000 + seed)
    gen = torch.Generator().manual_seed(2000 + seed)
    theta = new_policy(seed)
    opt = torch.optim.SGD([theta], lr=LR)
    ctx = torch.arange(N_CONTEXTS).unsqueeze(1).expand(N_CONTEXTS, G)
    coss = []
    for it in range(steps):
        a = sample_actions(theta, G, gen)
        r = reward_fn(ctx.numpy(), a.numpy(), rng)
        adv = (r - r.mean(1, keepdims=True)) / (r.std(1, keepdims=True) + EPS)
        phi = torch.tensor(adv, dtype=torch.float32)
        logp = policy_logprobs(theta).gather(1, a)
        loss = -(logp * phi).mean()
        opt.zero_grad(); loss.backward()
        coss.append(cosine(-theta.grad.detach(), true_gradient(theta)))
        opt.step()
    return float(np.mean(coss))

Gs = [2, 4, 8, 16, 32]
cos_by_G = [grpo_cos_for_G(G) for G in Gs]
plt.figure(figsize=(6, 4))
plt.plot(Gs, cos_by_G, "o-", color="tab:blue", lw=2)
plt.xscale("log", base=2); plt.xticks(Gs, [str(g) for g in Gs])
plt.xlabel("group size G"); plt.ylabel("mean cosine to true gradient")
plt.title("GRPO: bigger groups -> tighter baseline -> better gradient")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
for G, c in zip(Gs, cos_by_G):
    print(f"  G={G:2d}: mean cosine = {c:.4f}")

print(f"\nTOTAL NOTEBOOK ELAPSED: {time.time()-_t_start:.2f}s")


## Takeaways & things to try

* **Same gradient, different baseline.** REINFORCE, A2C-style value baseline, and
  GRPO are all $\mathbb{E}[\nabla\log\pi\cdot\Phi]$. Only $\Phi$ differs. The
  baseline is *legal* by the **EGLP lemma**: any action-independent $b(c)$ leaves
  the gradient unbiased and cuts variance.
* **GRPO removes the critic.** Instead of learning $V(c)$, it uses the **mean
  reward of a freshly-sampled group** as a per-prompt baseline, and divides by the
  group std to whiten the advantage. *The group is the baseline.* You saw it match
  (or beat) the value-baseline curve with **no value network**.
* **Variance is the real story.** The cosine-to-true-gradient plot shows why
  baselines matter — the learning-curve speedup is downstream of variance
  reduction. GRPO's variance **shrinks as $G$ grows** (Bonus plot).
* **Connection to PPO / RLVR.** Real GRPO wraps this advantage in a **PPO-style
  clipped importance ratio** $\min(\rho\hat A,\ \mathrm{clip}(\rho,1\pm\epsilon)\hat A)$
  against the sampling policy, and uses a verifier/reward-model score as $r$ — the
  **RLVR** setting. Our bandit is the stripped-down core of that loop
  (prompt → group of responses → scores → group-relative advantage).

**Things to try**
1. Vary `GROUP_SIZE` (e.g. 2 vs 32) and re-run — watch variance & curves change.
2. Crank `REWARD_NOISE` up — baseline-free REINFORCE degrades fastest.
3. Turn off the std-normalization in `grpo` (use just `r - mean`) — still
   unbiased, but step sizes vary across prompts.
4. Add a **PPO-style clipped ratio**: cache `logp_old` at sampling time, compute
   `ratio = exp(logp_new - logp_old)`, and use `min(ratio*Â, clip(ratio,1±0.2)*Â)`.
5. Swap tabular logits for a tiny MLP on one-hot contexts — the EGLP argument is
   unchanged.

**Related notes (Notion):**
[GRPO](https://app.notion.com/p/37f95008d76681499281fcd797cb3d1d) ·
[RLHF / PPO-for-LLMs](https://app.notion.com/p/37f95008d76681e4b32ad7fabc88968a)
